#166 Risk score серверов на пиковых нагрузках

Условно сейчас 2024-03-31 19:00:00. Рассмотрите окно «последний час» — с 2024-03-31 18:00:00 (включительно) до 2024-03-31 19:00:00 (не включительно).

Найдите серверы, у которых в этом окне максимальное cpu_pct превысило 70%. Для каждого такого сервера посчитайте risk_score по формуле: `0.4 × AVG(cpu_pct) + 0.3 × VAR_SAMP(mem_pct) + 0.2 × AVG(disk_gb) + 0.1 × AVG(net_mb)`


In [ ]:
SELECT 
    server_id
    ,ROUND(
        0.4 * AVG(cpu_pct) + 0.3 * VAR_SAMP(mem_pct) + 0.2 * AVG(disk_gb) + 0.1 * AVG(net_mb)
        , 2
    ) AS risk_score
FROM server_metrics
WHERE sample_time >= '2024-03-31 18:00:00' AND sample_time < '2024-03-31 19:00:00' AND cpu_pct > 70
GROUP BY server_id
ORDER BY risk_score DESC 

#158 Срез складских остатков по статусам

В таблице inventory для каждой физической единицы товара хранится её текущий status: продано, доступно, возвращено.

Напишите запрос, который для каждого товара из products вернёт product_name и три числа: units_sold, units_available, units_returned. Товары, для которых ни одной единицы ещё нет на складе, тоже должны попасть в выдачу — c нулями в соответствующих столбцах.

In [ ]:
SELECT 
    product_name
    ,COUNT(
        CASE WHEN i.status = 'Sold' THEN 1 END
    ) AS units_sold
    ,COUNT(
        CASE WHEN i.status = 'Available' THEN 1 END
    ) AS units_available
    ,COUNT(
        CASE WHEN i.status = 'Returned' THEN 1 END
    ) AS units_returned
FROM products AS p
LEFT JOIN inventory AS i ON i.product_id = p.product_id
GROUP BY product_name

#151 Количество клиентов с открытым счётом помесячно

В таблице accounts хранятся счета клиентов: дата открытия, дата закрытия (NULL, если счёт ещё действует). У одного клиента может быть несколько счетов.

В таблице calendar лежит полный список дат — поле calendar_date.

Для каждого месяца посчитайте количество клиентов, у которых был открытый счёт хотя бы один день этого месяца. Клиент считается имеющим открытый счёт в месяце M, если существует счёт со следующими свойствами:
- open_date не позже последнего дня M
- close_date IS NULL или close_date позже первого дня M

In [ ]:
SELECT 
    c.calendar_date AS month_date,
	COUNT(DISTINCT a.client_id) AS clients_count
FROM calendar c
JOIN accounts a ON a.open_date <= (c.calendar_date + INTERVAL '1 month' - INTERVAL '1 day')::date
	AND (a.close_date IS NULL OR a.close_date > c.calendar_date )
WHERE EXTRACT(DAY FROM c.calendar_date) = 1
GROUP BY c.calendar_date
ORDER BY c.calendar_date

#150 Клиенты с валютными счетами по месяцам

В таблице accounts хранятся счета клиентов: open_date — дата открытия, close_date — дата закрытия (NULL, если счёт ещё действует), val — код валюты (у рублёвых счетов значение 'RUB').

В таблице calendar лежит полный список дат — поле calendar_date.

Для каждого первого числа месяца посчитайте количество клиентов, у которых на этот день был хотя бы один действующий валютный счёт.

Счёт считается действующим на дату D, если open_date <= D и close_date > D

In [ ]:
WITH first_day_of_the_month AS (
    SELECT 
        calendar_date AS month_date
    FROM calendar
    WHERE EXTRACT(DAY FROM calendar_date) = 1
)

SELECT 
    fdm.month_date
    ,COUNT(DISTINCT client_id) AS currency_clients
FROM accounts AS a
JOIN first_day_of_the_month AS fdm ON a.open_date <= fdm.month_date 
    AND (a.close_date IS NULL OR a.close_date > fdm.month_date) AND val != 'RUB'
GROUP BY fdm.month_date
ORDER BY fdm.month_date ASC 

#148 Просмотры по типу монетизации за февраль

На каждый день за февраль 2026 года выведите количество просмотров отдельно по монетизации SVOD и AVOD.

In [ ]:
SELECT 
    cw.show_date::date AS show_date
    ,COUNT(
        CASE 
            WHEN c.paid_type = 'SVOD' THEN 1
        END 
    ) AS svod_views
    ,COUNT(
        CASE 
            WHEN c.paid_type = 'AVOD' THEN 1
        END 
    ) AS avod_views
FROM content_watch AS cw 
JOIN content AS c ON c.content_id = cw.content_id
WHERE EXTRACT(MONTH FROM cw.show_date) = 2 
    AND EXTRACT(YEAR FROM cw.show_date) = 2026
GROUP BY show_date::date

#136 Пользователи, смотревшие видео каждый день в январе

Дана таблица watched_content с информацией о просмотрах видео пользователями.

Выведите список пользователей (user_id), смотревших видео каждый день в январе 2026 года.

In [ ]:
SELECT 
    user_id
FROM watched_content
WHERE EXTRACT(MONTH FROM date) = 1 AND EXTRACT(YEAR FROM date) = 2026
GROUP BY user_id
HAVING COUNT(DISTINCT date) = 31


#135 Статистика просмотров видео по дням

Дана таблица watched_content с информацией о просмотрах видео пользователями. Каждая строка в таблице соответствует одному просмотру.

Посчитайте в одном запросе:
- количество просмотров  видео по дням
- размер аудитории  — уникальное количество пользователей по дням

In [ ]:
SELECT 
    date
    ,COUNT(*) AS views
    ,COUNT(DISTINCT user_id) AS audience
FROM watched_content
GROUP BY date
ORDER BY date ASC 

#117 Время доставки заказов

Напишите SQL-запрос, который вычисляет время доставки (в целых часах) для заказов, соответствующих следующим условиям:
- Заказ был принят 1 января 2024 года 
- Заказ не был отменен  или возвращен 
- Время доставки рассчитывается как разница между временем первого статуса Accepted и временем статуса Delivered для каждого подходящего заказа.
- Результат должен содержать только целое количество часов, без учета минут. 


In [ ]:
SELECT 
  o1.order_id
	,FLOOR(
		EXTRACT(EPOCH FROM (o2.status_datetime - o1.status_datetime)) / 3600
	) AS delivery_time_hours
FROM OrderStatus o1
JOIN OrderStatus o2 ON o1.order_id = o2.order_id
WHERE o1.status = 'Accepted'
	AND o1.status_datetime::date = '2024-01-01'
	AND o2.status = 'Delivered'
	AND o1.order_id NOT IN (
		SELECT DISTINCT order_id
		FROM OrderStatus
		WHERE status IN ('Canceled', 'Returned')
	)


#81 Гистограмма твитов за 2022 год

Вам дана таблица данных твитов Twitter. Напишите запрос, чтобы получить гистограмму твитов, опубликованных каждым пользователем в 2022 году. Выведите количество твитов на пользователя как группу и количество пользователей Twitter, которые попадают в эту группу.

In [ ]:
WITH tweets_by_user AS (
    SELECT 
        user_id
        ,COUNT(tweet_id) AS tweet_bucket
    FROM Tweets
    WHERE EXTRACT(YEAR FROM tweet_date) = 2022
    GROUP BY user_id
)

SELECT 
    tweet_bucket
    ,COUNT(*) AS users_num
FROM tweets_by_user
GROUP BY tweet_bucket

#84 Процент подтверждения регистрации

Новые пользователи TikTok регистрируются, используя свои адреса электронной почты. Они подтверждают свою регистрацию, переходя по ссылке в текстовом email сообщении об активации своих учетных записей. Пользователи могут получать несколько email сообщений для подтверждения учетной записи, пока они не подтвердят свою новую учетную запись.

Старшему аналитику интересно узнать процент подтверждения регистрации для конкретного списка пользователей из таблицы «emails». Напишите запрос, чтобы узнать процент подтверждения. 

Допущения:
- Аналитика интересует процент подтверждений конкретных пользователей из таблицы «emails», которая может включать не всех пользователей, которых потенциально можно найти в таблице «texts».
- Например, user_id=1 из таблицы «emails» может отсутствовать в таблице «texts» и наоборот.

«Confirmed» в «signup_action» означает, что пользователь подтвердил свою учетную запись и успешно завершил процесс регистрации.


In [ ]:
SELECT 
    ROUND(
	    COUNT(texts.email_id) / COUNT(DISTINCT emails.email_id)::numeric, 3
	) AS activation_rate
FROM emails
LEFT JOIN texts ON emails.email_id = texts.email_id AND texts.signup_action = 'Confirmed'